In [4]:
"""
Simple OCR extractor using only functions (no classes)
Save as: simple_ocr.py
"""

import cv2
import pytesseract
import pandas as pd
import re
import os
from datetime import datetime

# Global variables for mappings
DAY_MAPPING = {
    'monday': 1, 'tuesday': 2, 'wednesday': 3, 'thursday': 4,
    'friday': 5, 'saturday': 6, 'sunday': 7,
    'mon': 1, 'tue': 2, 'wed': 3, 'thu': 4, 'fri': 5, 'sat': 6, 'sun': 7
}

REASON_MAPPING = {
    'f': 'Facebook', 'fb': 'Facebook', 'facebook': 'Facebook',
    'm': 'Marketing', 'marketing': 'Marketing',
    'live': 'Livestream', 'livestream': 'Livestream',
    'r1': 'Repeat Customer', 'repeat': 'Repeat Customer',
    'r2': 'Referral', 'referral': 'Referral',
    'w': 'Walk-in', 'walk-in': 'Walk-in', 'walkin': 'Walk-in'
}

def extract_text_from_image(image_path):
    """Extract text from image using OCR"""
    try:
        print(f"Processing image: {os.path.basename(image_path)}")
        
        # Read image
        img = cv2.imread(image_path)
        if img is None:
            print(f"Error: Could not read image {image_path}")
            return ""
        
        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Apply denoising
        denoised = cv2.fastNlMeansDenoising(gray)
        
        # Apply threshold
        _, thresh = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        # Extract text
        custom_config = r'--oem 3 --psm 6'
        text = pytesseract.image_to_string(thresh, config=custom_config)
        
        print(f"Text extracted successfully ({len(text)} characters)")
        return text
        
    except Exception as e:
        print(f"Error extracting text: {e}")
        return ""

def parse_date_and_day(text):
    """Extract date and day from text"""
    # Look for date pattern
    date_pattern = r'Date\s*:?\s*(\d{1,2})/(\d{1,2})/(\d{4})\s*\(([^)]+)\)'
    match = re.search(date_pattern, text, re.IGNORECASE)
    
    if match:
        day, month, year, day_text = match.groups()
        
        # Format date
        try:
            date_obj = datetime.strptime(f"{day}/{month}/{year}", "%d/%m/%Y")
            formatted_date = date_obj.strftime("%Y-%m-%d")
        except:
            formatted_date = f"{year}-{month.zfill(2)}-{day.zfill(2)}"
        
        # Get day number
        day_num = None
        day_text_clean = day_text.strip().lower()
        for day_key, num in DAY_MAPPING.items():
            if day_key in day_text_clean:
                day_num = num
                break
        
        return formatted_date, day_text.title(), day_num
    
    return None, None, None

def extract_reason_code(line):
    """Extract reason code from a line of text"""
    # Look for reason code patterns
    reason_patterns = [
        r'\b(F|FB|M|LIVE|R1|R2|W)\b',
        r'\b(facebook|marketing|livestream|repeat|referral|walk-?in)\b'
    ]
    
    for pattern in reason_patterns:
        matches = re.findall(pattern, line, re.IGNORECASE)
        if matches:
            code = matches[0].lower().strip()
            if code in REASON_MAPPING:
                return REASON_MAPPING[code]
            else:
                return f"UNCLEAR: {code}"
    
    return "Not specified"

def parse_sales_data(text):
    """Parse sales entries from text"""
    lines = text.split('\n')
    sales_data = []
    unclear_markings = set()
    
    print(f"Parsing {len(lines)} lines...")
    
    for i, line in enumerate(lines):
        line = line.strip()
        if not line or len(line) < 10:
            continue
        
        # Skip header lines
        if any(header in line.lower() for header in ['s/n', 'name', 'sales order', 'amount', 'showroom']):
            continue
        
        # Look for sales data pattern
        if re.search(r'1-\d+', line) and re.search(r'\d+', line):
            print(f"Processing line {i}: {line}")
            
            # Extract components
            name = None
            sales_order = None
            amount = None
            
            # Find sales order (pattern: 1-xxxxxx)
            sales_order_match = re.search(r'1-\d+', line)
            if sales_order_match:
                sales_order = sales_order_match.group()
            
            # Find amount (numbers, possibly with $ or commas)
            amount_matches = re.findall(r'\$?[\d,]+\.?\d*', line)
            for amt in amount_matches:
                clean_amt = re.sub(r'[^\d.]', '', amt)
                if len(clean_amt) >= 2:  # Reasonable amount
                    amount = clean_amt
                    break
            
            # Find name (alphabetic words)
            words = line.split()
            for word in words:
                if word.isalpha() and len(word) > 2 and word.upper() not in ['DATE', 'NOVA', 'TRADEHUB']:
                    name = word.upper()
                    break
            
            # Extract reason code
            reason_code = extract_reason_code(line)
            if "UNCLEAR:" in reason_code:
                unclear_markings.add(reason_code.replace("UNCLEAR: ", ""))
            
            # Add record if we have minimum required data
            if name and sales_order:
                record = {
                    'name': name,
                    'sales_order_no': sales_order,
                    'amount': amount or "0.00",
                    'reason_code': reason_code
                }
                sales_data.append(record)
                print(f"  → Extracted: {record}")
    
    return sales_data, unclear_markings

def process_image(image_path):
    """Process a single image"""
    print(f"\n{'='*60}")
    print(f"Processing: {os.path.basename(image_path)}")
    print(f"{'='*60}")
    
    # Extract text
    text = extract_text_from_image(image_path)
    if not text:
        print("No text extracted")
        return []
    
    # Show text preview
    print(f"Text preview (first 200 chars):\n{text[:200]}...\n")
    
    # Parse date and day
    date, day_text, day_num = parse_date_and_day(text)
    print(f"Date: {date}")
    print(f"Day: {day_text} ({day_num})")
    
    # Parse sales data
    sales_data, unclear_markings = parse_sales_data(text)
    
    # Add date info to records
    for record in sales_data:
        record['date'] = date or "Unknown"
        record['day'] = day_text or "Unknown"
        record['day_number'] = day_num or 0
        record['source_file'] = os.path.basename(image_path)
    
    print(f"\nExtracted {len(sales_data)} records")
    
    if unclear_markings:
        print(f"Unclear markings: {unclear_markings}")
    
    return sales_data

def export_to_csv(data, output_file):
    """Export data to CSV"""
    if not data:
        print("No data to export")
        return False
    
    try:
        df = pd.DataFrame(data)
        
        # Reorder columns
        columns = ['date', 'day', 'day_number', 'name', 'sales_order_no', 'amount', 'reason_code', 'source_file']
        existing_columns = [col for col in columns if col in df.columns]
        df = df[existing_columns]
        
        df.to_csv(output_file, index=False)
        print(f"Data exported to: {output_file}")
        return True
        
    except Exception as e:
        print(f"Error exporting CSV: {e}")
        return False

def main():
    """Main function"""
    print("Simple Sales Report OCR Extractor")
    print("=" * 50)
    
    # Get image path
    image_path = input("Enter the path to your sales report image: ").strip()
    
    # Remove quotes if present
    if image_path.startswith('"') and image_path.endswith('"'):
        image_path = image_path[1:-1]
    
    if not os.path.exists(image_path):
        print(f"Error: File '{image_path}' not found")
        return
    
    # Process image
    data = process_image(image_path)
    
    # Show results
    print(f"\n{'='*60}")
    print("RESULTS")
    print(f"{'='*60}")
    
    if data:
        print(f"Successfully extracted {len(data)} records:")
        for i, record in enumerate(data, 1):
            print(f"{i}. {record}")
        
        # Export option
        export = input(f"\nExport to CSV? (y/n): ").strip().lower()
        if export == 'y':
            output_file = input("Enter CSV filename (default: extracted_data.csv): ").strip()
            if not output_file:
                output_file = "extracted_data.csv"
            if not output_file.endswith('.csv'):
                output_file += '.csv'
            
            export_to_csv(data, output_file)
    else:
        print("No sales data could be extracted from the image")

if __name__ == "__main__":
    main()

Simple Sales Report OCR Extractor

Processing: photo_2025-09-10_09-33-52.jpg
Processing image: photo_2025-09-10_09-33-52.jpg
Error extracting text: tesseract is not installed or it's not in your PATH. See README file for more information.
No text extracted

RESULTS
No sales data could be extracted from the image
